# 🛰️ Explainable Satellite Image Classification for Deforestation Monitoring
**End-to-End Pipeline**: EuroSAT Land Cover Classification • Grad-CAM Explainability • Temporal Change Detection • Gradio Web App

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 1. Setup Environment & Install Dependencies

In [ ]:
# Verify GPU availability
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"Available GPUs: {gpus if gpus else 'No GPU found (Running on CPU)'}")

# Install dependencies
!pip install -q gradio opencv-python-headless scikit-learn matplotlib seaborn

## 2. Download EuroSAT Dataset & Build Data Pipelines

In [ ]:
import os
import urllib.request
import zipfile
from pathlib import Path

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
zip_path = DATA_DIR / "EuroSAT.zip"

if not (DATA_DIR / "2750").exists():
    print("Downloading EuroSAT dataset (~90 MB)...")
    url = "https://madm.dfki.de/files/sentinel/EuroSAT.zip"
    opener = urllib.request.build_opener()
    opener.addheaders = [("User-Agent", "Mozilla/5.0")]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(url, zip_path)
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Extraction complete!")
else:
    print("EuroSAT dataset already present.")

## 3. Build & Train Transfer Learning Model (EfficientNetB0)

In [ ]:
# Import project modules
import sys
sys.path.append(".")

from src.data_loader import build_data_pipelines, EUROSAT_CLASSES
from src.model import build_transfer_model, unfreeze_top_layers
from src.train import train_model

# Train with 5 warmup epochs + 5 fine-tuning epochs for quick Colab execution
model = train_model(
    data_dir="data/raw/2750",
    backbone="EfficientNetB0",
    batch_size=32,
    epochs_stage1=5,
    epochs_stage2=5,
    max_samples=5000  # Remove or increase for full 27,000 dataset run
)

## 4. Grad-CAM Explainability Demo

In [ ]:
import matplotlib.pyplot as plt
from src.gradcam import GradCAM
from src.data_loader import create_synthetic_samples_if_needed

samples = create_synthetic_samples_if_needed()
gradcam = GradCAM(model)

# Run Grad-CAM on a forest patch
res = gradcam.generate_overlay(samples["Forest"], target_size=(128, 128))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(res["original"])
axes[0].set_title("Original Satellite Patch")
axes[0].axis("off")

axes[1].imshow(res["heatmap_colored"])
axes[1].set_title("Grad-CAM Heatmap")
axes[1].axis("off")

axes[2].imshow(res["overlay"])
axes[2].set_title(f"Superimposed (Prediction: {res['predicted_class']})")
axes[2].axis("off")

plt.show()
print(f"Top 3 Predictions: {res['top3']}")

## 5. Temporal Change Detection Demo

In [ ]:
from src.change_detection import TemporalDeforestationDetector

detector = TemporalDeforestationDetector(model)
change_res = detector.detect_change(samples["temporal_t1"], samples["temporal_t2"])

print(f"Status: {change_res['severity']}")
print(f"Transition: {change_res['transition_type']}")
print(f"Canopy Loss Percentage: {change_res['canopy_loss_percent']}%")

plt.figure(figsize=(12, 4))
plt.imshow(change_res["composite"])
plt.title("Temporal Tri-View: [T1 Baseline | T2 Post-Event | Deforestation Overlay]")
plt.axis("off")
plt.show()

## 6. Launch Interactive Gradio Web App with Public URL

In [ ]:
from src.app import create_app

app = create_app()
# share=True creates a public gradio.live URL in Google Colab
app.launch(share=True)